## Setup the environment

In [1]:
%%capture
%pip install transformers datasets evaluate sacrebleu torch huggingface_hub
%pip install accelerate sentencepiece tiktoken peft bitsandbytes safetensors trl

In [1]:
import torch
import warnings
from huggingface_hub import notebook_login

warnings.filterwarnings(
    "ignore",
    message=r"MatMul8bitLt: inputs will be cast from .* to float16 during quantization"
)

notebook_login()

if torch.cuda.is_available():
    print(f"{torch.cuda.device_count()} GPU(s) detected! Device ID: {torch.cuda.current_device()}")
    print(torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print(f"No GPU detected! Device ID: {torch.cuda.current_device()}")
    device = -1

1 GPU(s) detected! Device ID: 0
Tesla T4
BF16 supported: True


## Load Dataset

In [2]:
from datasets import load_dataset

data = load_dataset("kalixlouiis/Myanmar-English-general-text-translation")

## Preprocess The Dataset

In [3]:
from transformers import NllbTokenizer

checkpoint = "Emilio407/nllb-200-1.3B-8bit"
tokenizer = NllbTokenizer.from_pretrained(
    checkpoint,
    src_lang="my_MM",
    tgt_lang="en_Latn",
)

In [4]:
def preprocess_function(examples):
    inputs = examples['my']
    targets = examples["en"]
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=128,
        truncation=True,
        padding=False
    )
    return model_inputs

In [5]:
tokenized_data = data.map(preprocess_function, remove_columns=data["train"].column_names)
tokenized_data

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 9616
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1202
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1203
    })
})

In [6]:
from torch.utils.data import Subset

indices = list(range(1000))
tokenized_train_data = Subset(tokenized_data['train'], list(range(100)))
tokenized_val_data = Subset(tokenized_data['validation'], list(range(10)))
tokenized_test_data = Subset(tokenized_data['test'], list(range(10)))

## Model Configuration

In [7]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
   load_in_4bit=True,
   bnb_4bit_quant_type="nf4",
   bnb_4bit_use_double_quant=True,
   bnb_4bit_compute_dtype=torch.float32
)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint, device_map="cuda:0", quantization_config=bnb_config)

/usr/local/lib/python3.13/dist-packages/transformers/quantizers/auto.py:275: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

In [8]:
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    # the rank of the adapter, the lower the fewer parameters you'll need to train
    r=8,
    lora_alpha=16, # multiplier, usually 2*r
    bias="none",
    lora_dropout=0.05,
    task_type="SEQ_2_SEQ_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'out_proj', 'fc1', 'fc2'],
)
model = get_peft_model(model, config)

## Preparation to fine tune the model

In [9]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [10]:
import evaluate
import numpy as np

metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

## Fine-tuning

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="nlbb",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=2,
    predict_with_generate=True,
    fp16=True, #change to bf16=True for XPU
    push_to_hub=True,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_data,
    eval_dataset=tokenized_val_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,2.536201,6.554500,20.000000


Streaming output truncated to the last 5000 lines.


Epoch,Training Loss,Validation Loss,Bleu,Gen Len
1,No log,2.536201,6.554500,20.000000
2,No log,2.549223,5.834500,21.900000


TrainOutput(global_step=26, training_loss=2.7488937377929688, metrics={'train_runtime': 139.2489, 'train_samples_per_second': 1.436, 'train_steps_per_second': 0.187, 'total_flos': 42984185167872.0, 'train_loss': 2.7488937377929688, 'epoch': 2.0})

In [ ]:
trainer.push_to_hub()

CommitInfo(commit_url='https://huggingface.co/minkhantycc/nlbb/commit/fbab0118b88575c6a0574cac8c62d7b58ddcba1a', commit_message='End of training', commit_description='', oid='fbab0118b88575c6a0574cac8c62d7b58ddcba1a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/minkhantycc/nlbb', endpoint='https://huggingface.co', repo_type='model', repo_id='minkhantycc/nlbb'), pr_revision=None, pr_num=None)

## Testing and Evaluation

In [11]:
from transformers import AutoTokenizer
text = "အိမ်မှာ ဝိုင်ဖိုင်ပျက်သွားလို့ လာကြည့်ပေးပါ"
tokenizer = AutoTokenizer.from_pretrained("minkhantycc/nlbb")
tokenizer.src_lang ="my_MM"
inputs = tokenizer(text, return_tensors="pt").to("cuda:0")

tokenizer_config.json:   0%|          | 0.00/3.72k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [12]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("minkhantycc/nlbb", device_map="cuda:0")
outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("en_Latn"), max_length=30,
)
tokenizer.batch_decode(outputs, skip_special_tokens=True)

Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/768 [00:00<?, ?it/s]

['Wifi broke at home, come and check it out.  Please come and check it out.']

## Comparison of Models (Base VS Fine-tuned)

In [13]:
from transformers import AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

### Evaluation of Base Model

In [14]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    device_map="cuda:0",
    torch_dtype=torch.float16,
)
base_model.eval()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=base_model,
)
test_loader = DataLoader(
    tokenized_test_data,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
)
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels")
        inputs = {
            k: v.to(base_model.device)
            for k, v in batch.items()
        }
        generated_tokens = base_model.generate(
            **inputs,
            max_new_tokens=128,
        )
        all_predictions.extend(
            generated_tokens.cpu().numpy()
        )
        all_labels.extend(
            labels.cpu().numpy()
        )

predictions = pad_sequence(
    [torch.tensor(x) for x in all_predictions],
    batch_first=True,
    padding_value=tokenizer.pad_token_id,
).numpy()

labels = pad_sequence(
    [torch.tensor(x) for x in all_labels],
    batch_first=True,
    padding_value=-100,
).numpy()

baseline_results = compute_metrics(
    (predictions, labels)
)

print("Baseline results:")
print(baseline_results)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Baseline results:
{'bleu': 6.4056, 'gen_len': np.float64(19.3)}


### Evaluation of Finetuned Model

In [15]:
finetuned_model = AutoModelForSeq2SeqLM.from_pretrained(
    "minkhantycc/nlbb",
    device_map="cuda:0",
    torch_dtype=torch.float16,
)

finetuned_model.eval()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=finetuned_model,
)

test_loader = DataLoader(
    tokenized_test_data,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
)

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        labels = batch.pop("labels")

        inputs = {
            k: v.to(finetuned_model.device)
            for k, v in batch.items()
        }

        generated_tokens = finetuned_model.generate(
            **inputs,
            max_new_tokens=128,
        )

        all_predictions.extend(
            generated_tokens.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

predictions = pad_sequence(
    [torch.tensor(x) for x in all_predictions],
    batch_first=True,
    padding_value=tokenizer.pad_token_id,
).numpy()

labels = pad_sequence(
    [torch.tensor(x) for x in all_labels],
    batch_first=True,
    padding_value=-100,
).numpy()

finetuned_results = compute_metrics(
    (predictions, labels)
)

print("Finetuned results:")
print(finetuned_results)

Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/768 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Finetuned results:
{'bleu': 6.4576, 'gen_len': np.float64(19.4)}
